# Phase 2: The Baseline Classifier (Naive Bayes)

## 🏗 Overview: The "Fast & Logical" First Model
In this phase, we build our initial classifier using **Multinomial Naive Bayes**. 

###  Why Naive Bayes?
1.  **Transparency (The White Box)**: This model is based on pure probability (counting word frequencies). We can see exactly which words define Fake vs Real news, making it perfect for our final presentation.
2.  **Benchmark Strategy**: Before trying complex "Black Box" Neural Networks, we must set a benchmark. If a simple model performs exceptionally well, we save the company (or instructors) time and resources.
3.  **Speed**: It is lightning-fast, allowing us to iterate quickly even on large datasets like this (~40,000 articles).

In [ ]:
import os
import pandas as pd
import joblib
import nltk

# Colab Integration: Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # ⚠️ Use exact same folder name as in your Drive
    BASE_PATH = '/content/drive/MyDrive/Project 2/project-nlp-challenge'
    print("✅ Running in Colab. Google Drive mounted.")
except ImportError:
    BASE_PATH = '.'
    print("💻 Running Locally.")

# NLTK Downloads for ephemeral environments
nltk.download('stopwords')
nltk.download('punkt')

# Ensure directories exist
os.makedirs(os.path.join(BASE_PATH, 'models'), exist_ok=True)
os.makedirs(os.path.join(BASE_PATH, 'dataset'), exist_ok=True)


## 1. Data Ingestion (The Sliced Ingredients)
We load our stratified Train and Test sets created in Phase 1.

In [ ]:
train_df = pd.read_csv(os.path.join(BASE_PATH, 'dataset/train.csv'))
test_df = pd.read_csv(os.path.join(BASE_PATH, 'dataset/test.csv'))

# Load the fitted vectorizer
tfidf = joblib.load(os.path.join(BASE_PATH, 'models/vectorizer.joblib'))

print(f"Data Loaded. Train: {train_df.shape}, Test: {test_df.shape}")

## 2. Transformation
We convert the cleaned text into their TF-IDF numerical vectors.

In [ ]:
X_train = tfidf.transform(train_df['cleaned_text'].astype(str))
y_train = train_df['label']

X_test = tfidf.transform(test_df['cleaned_text'].astype(str))
y_test = test_df['label']

print("✅ Vectors Generated.")

## 3. Training: The Multinomial Naive Bayes
We train the model on our training data.

In [ ]:
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

print("✅ Model Trained!")

### 🏆 Double Training: NB & Logistic Regression
We train both models to compare performance in the next phase.

In [ ]:
from sklearn.linear_model import LogisticRegression

# 1. Train Naive Bayes (Standard)
nb_model.fit(X_train, y_train)
joblib.dump(nb_model, os.path.join(BASE_PATH, 'models/nb_tfidf_classifier.joblib'))

# 2. Train Logistic Regression (Required for comparison phase)
lr_tfidf_model = LogisticRegression(max_iter=1000)
lr_tfidf_model.fit(X_train, y_train)
joblib.dump(lr_tfidf_model, os.path.join(BASE_PATH, 'models/lr_tfidf_classifier.joblib'))

print("✅ Both Models (NB & LR) Trained and Persistent.")

### 🏆 Persistence & Comparison Training
We save our model and train a secondary Logistic Regression to compare in the next phase.


## 4. Evaluation & Visualization
How well does our model perform on the 'Final Exam' (The test set)?

In [ ]:
y_pred = nb_model.predict(X_test)

print("📊 Classification Report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Fake', 'Real'], yticklabels=['Fake', 'Real'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

## 5. Final Deliverable: Validation Predictions
we must predict labels for `validation_data.csv` and ensure the output file has **NO extra columns** and maintains the exact original format.

In [ ]:
val_df = pd.read_csv(os.path.join(BASE_PATH, 'dataset/validation_data.csv'))

# 1. Applying exactly the same fusion and cleaning logic
# Note: In a production setting, we would import the clean_text function from a module.
def clean_text(text):
    import re, string
    from nltk.tokenize import word_tokenize
    from nltk.corpus import stopwords
    if pd.isna(text): return ""
    text = text.lower()
    pattern = re.compile('[%s]' % re.escape(string.punctuation))
    text = pattern.sub('', text)
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [w for w in tokens if w not in stop_words]
    return " ".join(filtered_tokens)

print("Scrubbing validation data...")
val_full_text = (val_df['title'] + " " + val_df['text']).apply(clean_text)

# 2. Vectorization
X_val = tfidf.transform(val_full_text)

# 3. Prediction
val_df['label'] = nb_model.predict(X_val)

# 4. Final Guard: Export strictly following original format
# Columns required: original columns only, no 'full_text' or 'cleaned_text' helper columns
final_output_path = os.path.join(BASE_PATH, 'dataset/validation_results.csv')
val_df.to_csv(final_output_path, index=False)

print(f"✅ Validation predictions saved to: {final_output_path}")
print("Format Verified: Exactly like validation_data.csv.")